In [ ]:
import eos
import os
import numpy as np
import matplotlib.pyplot as plt
from wquantiles import quantile_1D
import ast
import cmath
import re
import pandas as pd
from scipy.stats import chi2

colors = eos.figure.ItemColorCycler()._colors

resample = False

In [ ]:
BASE_DIRECTORY = os.getcwd()
if not os.path.exists(BASE_DIRECTORY):
    os.makedirs(os.path.abspath(BASE_DIRECTORY), exist_ok=True)

In [ ]:
def corner_plot(path):
    figure_args = {
        'type': 'corner',
        'contents': [{
            'path': path,
            'label': 'Global fit'
        }]
    }
    figure = eos.figure.FigureFactory.from_dict(**figure_args)
    figure.draw()

In [ ]:
def wstd(samples, weights=None, axis=0):
    mean_w = np.average(samples, weights=weights, axis=axis)
    return np.sqrt(np.average((samples - mean_w)**2, weights=weights, axis=axis))

def get_gof(analysis, param_array):
    parameter_list = []

    for param in analysis.init_args['priors']:
        if param['type'] == 'transform':
            parameter_list += param['parameters']
        else:
            parameter_list.append(param['parameter'])

    for i, param in enumerate(parameter_list):
        analysis.parameters.set(param, param_array[i])

    return analysis.goodness_of_fit()

In [ ]:
mpi    = eos.Parameters()["mass::pi^+"].evaluate()
sPlus  = 4 * mpi**2
sMinus = 0.0
sIn    = 0.850509 #pi-omega threshold

In [ ]:
def a11_calculator(sl_params):
    r2, r3, _, _ = sl_params

    coefficient = (
        -0.36294823212268046 * r2
        +0.06946304407045559 * r3
    )

    return sPlus * coefficient.real


def b11_calculator(sl_params):
    r2, r3, r4, r5 = sl_params

    coefficient = (
         0.03311321577857029   * r2
        -0.23706596387010614   * r2**2
        -0.14455309077802547   * r3
        +0.04537099795088154   * r2*r3
        +0.07902198795670203   * r4
        -0.00453709979508815   * r5
    )

    return sPlus**2 * coefficient.real

def calculate_sl_values(sl_params):
    a11_values = (
        np.array(list(map(a11_calculator, sl_params.samples))))

    b11_values = (
        np.array(list(map(b11_calculator, sl_params.samples))))

    return a11_values, b11_values

def summarize_sl_values(name, a11_values, b11_values, weights):
    a11_mean = np.average(a11_values, weights=weights)
    a11_std  = wstd(a11_values, weights=weights)

    b11_mean = np.average(b11_values, weights=weights)
    b11_std  = wstd(b11_values, weights=weights)

    print(f"{name} a11 = {a11_mean:.4f} ± {a11_std:.4f}")
    print(f"{name} b11 = {b11_mean:.4f} ± {b11_std:.4f}")

    return a11_mean, a11_std, b11_mean, b11_std

sl_parameters = eos.Parameters()

a11_parameter = sl_parameters.declare_and_insert(
    'derived::a11',
    r'a_{11}',
    eos.Unit.Unity(),
    0.0,
    -100.0,
    100.0
)

b11_parameter = sl_parameters.declare_and_insert(
    'derived::b11',
    r'b_{11}',
    eos.Unit.Unity(),
    0.0,
    -100.0,
    100.0
)


def save_sl_samples(posterior, a11_values, b11_values, weights):
    sl_values = np.column_stack((
        a11_values,
        b11_values
    ))

    output_path = os.path.join(
        BASE_DIRECTORY,
        'data',
        posterior,
        'scattering-lengths',
        'samples'
    )

    os.makedirs(
        os.path.dirname(output_path),
        exist_ok=True
    )

    eos.data.ImportanceSamples.create(
        output_path,
        [a11_parameter, b11_parameter],
        sl_values,
        weights
    )

    return output_path


# Sample independent Gaussian literature values (these values are from table 2 of arxiv:hep-ph/013088v1).
rng = np.random.default_rng(42)
n_samples = 2000

a11_theory = rng.normal(0.0379, 0.0005, n_samples)
b11_theory = rng.normal(0.00567, 0.00013, n_samples)

# All sampled curves have equal weight.
theory_weights = np.full(n_samples, 1.0 / n_samples)

# Physical s values, from threshold to 0.2 GeV^2.
s_theory = np.linspace(sPlus, 0.2, 200)
eps = s_theory - sPlus

# Each row is one complete curve from one sampled (a11, b11) pair.
t_theory_samples = eps[None, :] / sPlus * ( a11_theory[:, None] + b11_theory[:, None] * eps[None, :] / sPlus)

# Save the parameter samples, curve samples, grid, and weights.
theory_path = os.path.join(BASE_DIRECTORY, 'data', 'theory', 't11-threshold-samples.npz')
os.makedirs(os.path.dirname(theory_path), exist_ok=True)

np.savez_compressed(theory_path, s=s_theory, a11=a11_theory, b11=b11_theory, t11=t_theory_samples, weights=theory_weights)

In [ ]:
def charge_radius_squared_calculator(fprime_values):
    # 6 * df/dpsi at psi(0) * dpsi/ds at s=0 * (hbar c)^2
    return 6 * fprime_values * (-1.31241754914090000000) * (0.19732697178003900000)**2 #fm^2

In [ ]:
def get_f_rho(sample, ana):
    get_gof(ana, sample)
    s_rho = (sample[0] - 1j * sample[1] / 2)**2
    return [eos.Observable.make("0->pipi::Re{f_+}(Re{q2},Im{q2})",
                        ana.parameters,
                        eos.Kinematics({"Re{q2}": np.real(s_rho), "Im{q2}": np.imag(s_rho)}),
                        eos.Options({'form-factors': "BHKMNR2026", "n-resonances": '1'})).evaluate(),
            eos.Observable.make("0->pipi::Im{f_+}(Re{q2},Im{q2})",
                        ana.parameters,
                        eos.Kinematics({"Re{q2}": np.real(s_rho), "Im{q2}": np.imag(s_rho)}),
                        eos.Options({'form-factors': "BHKMNR2026", "n-resonances": '1'})).evaluate()]

def get_f_bfp(ana, bfp, q2):
    get_gof(ana, bfp)
    return eos.Observable.make("0->pipi::Abs{f_+}^2(q2)",
                        ana.parameters,
                        eos.Kinematics({'q2': q2}),
                        eos.Options({'form-factors': "BHKMNR2026", "n-resonances": '1'})).evaluate()

def get_f_arg_bfp(ana, bfp, q2):
    get_gof(ana, bfp)
    return eos.Observable.make("0->pipi::Arg{f_+}(q2)",
                        ana.parameters,
                        eos.Kinematics({'q2': q2}),
                        eos.Options({'form-factors': "BHKMNR2026", "n-resonances": '1'})).evaluate()

def get_f_arg_of_psi_bfp(ana, bfp, psi):
    get_gof(ana, bfp)
    return eos.Observable.make("0->pipi::Arg{f_+}(Re{psi},Im{psi})@BHKMNR2026",
                        ana.parameters,
                        eos.Kinematics({'Re{psi}': np.real(psi), 'Im{psi}': np.imag(psi)}),
                        eos.Options({'form-factors': "BHKMNR2026", "n-resonances": '1'})).evaluate()

def get_f_abs_of_psi_bfp(ana, bfp, psi):
    get_gof(ana, bfp)
    return eos.Observable.make("0->pipi::Abs{f_+}^2(Re{psi},Im{psi})@BHKMNR2026",
                        ana.parameters,
                        eos.Kinematics({'Re{psi}': np.real(psi), 'Im{psi}': np.imag(psi)}),
                        eos.Options({'form-factors': "BHKMNR2026", "n-resonances": '1'})).evaluate()

In [ ]:
COUPLING_COLUMNS = [
    ("derived::Re{g_rho_gamma}", r"$\mathrm{Re}\,g_{\rho\gamma}$"),
    ("derived::Im{g_rho_gamma}", r"$\mathrm{Im}\,g_{\rho\gamma}$"),
    ("derived::Re{g_rho_pi_pi}", r"$\mathrm{Re}\,g_{\rho\pi\pi}$"),
    ("derived::Im{g_rho_pi_pi}", r"$\mathrm{Im}\,g_{\rho\pi\pi}$"),
]

coupling_parameters = eos.Parameters()

for name, latex in COUPLING_COLUMNS:
    if not coupling_parameters.has(name):
        coupling_parameters.declare_and_insert(
            name, latex, eos.Unit.Unity(),
            0.0, -100.0, 100.0,
        )

coupling_parameter_list = [
    coupling_parameters[name] for name, _ in COUPLING_COLUMNS
]

def residue_file_tag(posterior):
    match = re.fullmatch(
        r"FF-fp-I1-order(\d+)-(together|without-Belle)",
        posterior,
    )
    if match is None:
        raise ValueError(f"Unexpected posterior: {posterior}")

    order, fit_type = match.groups()

    if fit_type == "together":
        return f"together{order}"
    return f"wBelle{order}"

def export_rho_residues_psi(posterior, recompute=False):
    """Create residues_psi_<file_tag>.wl for Mathematica."""
    file_tag = residue_file_tag(posterior)
    fit_directory = os.path.join(BASE_DIRECTORY, "data", posterior)
    fit = eos.data.ImportanceSamples(
        os.path.join(fit_directory, "samples")
    )

    residue_parts = []

    for part in ("re", "im"):
        prediction_name = f"{part}-rho-residue-psi"
        prediction_path = os.path.join(
            fit_directory, f"pred-{prediction_name}"
        )

        if recompute or not os.path.exists(prediction_path):
            eos.tasks.predict_observables(
                af, posterior, prediction_name,
                base_directory=BASE_DIRECTORY,
            )

        prediction = eos.data.Prediction(prediction_path)
        residue_parts.append(prediction.samples[:, 0])

    # Re(residue_psi), Im(residue_psi), M, Gamma
    data = np.column_stack((
        residue_parts[0],
        residue_parts[1],
        fit.samples[:, :2],
    ))

    if not np.isfinite(data).all():
        raise ValueError(f"{posterior}: non-finite WL input")

    body = ",\n".join(
        f"{{{r:.17g}, {i:.17g}, {m:.17g}, {g:.17g}}}"
        for r, i, m, g in data
    ).replace("e", "*^")

    wl_path = os.path.join(
        BASE_DIRECTORY, f"residues_psi_{file_tag}.wl"
    )
    with open(wl_path, "w", encoding="utf-8") as file:
        file.write("{\n" + body + "\n}")

    print(f"Wrote {wl_path} ({len(data)} rows)")
    return wl_path


def save_rho_couplings(posterior):
    """Read residues_s_<file_tag>.csv and save the four coupling columns."""
    file_tag = residue_file_tag(posterior)
    fit_directory = os.path.join(BASE_DIRECTORY, "data", posterior)
    fit = eos.data.ImportanceSamples(
        os.path.join(fit_directory, "samples")
    )

    csv_path = os.path.join(
        BASE_DIRECTORY, f"residues_s_{file_tag}.csv"
    )
    residue_data = np.loadtxt(
        csv_path, delimiter=",", ndmin=2
    )

    if residue_data.shape != (len(fit.samples), 2):
        raise ValueError(
            f"{csv_path}: expected shape {(len(fit.samples), 2)}, "
            f"got {residue_data.shape}"
        )

    res_s = residue_data[:, 0] + 1j * residue_data[:, 1]
    M = fit.samples[:, 0]
    Gamma = fit.samples[:, 1]
    s_rho = (M - 1j * Gamma / 2)**2

    print(f"{posterior}: evaluating couplings...", flush=True)

    analysis = af.analysis(posterior)
    f_components = np.asarray([
        get_f_rho(sample, analysis) for sample in fit.samples
    ])
    F_I = f_components[:, 0] + 1j * f_components[:, 1]

    sigma = np.sqrt(1 - sPlus / s_rho)
    g_rho_gamma = np.sqrt(
        24j * np.pi * s_rho / (F_I * res_s * sigma**3)
    )
    g_rho_pi_pi = -(res_s / s_rho) * g_rho_gamma

    values = np.column_stack((
        g_rho_gamma.real,
        g_rho_gamma.imag,
        g_rho_pi_pi.real,
        g_rho_pi_pi.imag,
    ))

    if not np.isfinite(values).all():
        raise ValueError(f"{posterior}: non-finite couplings")

    for parameter, column in zip(coupling_parameter_list, values.T):
        parameter.set_min(float(column.min()))
        parameter.set_max(float(column.max()))

    couplings_path = os.path.join(
        fit_directory, "rho-couplings", "samples"
    )
    os.makedirs(os.path.dirname(couplings_path), exist_ok=True)

    eos.data.ImportanceSamples.create(
        couplings_path,
        coupling_parameter_list,
        values,
        fit.weights,
    )

    print(f"Saved {len(values)} samples to {couplings_path}")
    return couplings_path


def show_rho_couplings(posterior):
    """Print the mean and standard deviation of both saved complex couplings."""
    couplings_path = os.path.join(
        BASE_DIRECTORY, "data", posterior,
        "rho-couplings", "samples",
    )
    saved = eos.data.ImportanceSamples(couplings_path)

    means = np.average(
        saved.samples, weights=saved.weights, axis=0
    )
    stds = np.sqrt(np.average(
        (saved.samples - means)**2,
        weights=saved.weights,
        axis=0,
    ))

    for quantity, index in (
        ("g_rho_gamma", 0),
        ("g_rho_pi_pi", 2),
    ):
        print(
            f"{posterior}: {quantity} = "
            f"({means[index]:.4f} ± {stds[index]:.4f}) "
            f"+ i * ({means[index + 1]:.4f} ± "
            f"{stds[index + 1]:.4f})"
        )

In [ ]:
def show_results_table(posterior):
    match = re.fullmatch(r'FF-fp-I1-order(\d+)-(.+)', posterior)
    if match is None:
        raise ValueError(f'Unexpected posterior name: {posterior}')

    order, fit_type = match.groups()
    fit_name = {
        'together': 'Together',
        'without-Belle': 'Without Belle',
    }.get(fit_type, fit_type.replace('-', ' '))
    label = f'{fit_name} order {order}'

    fit_directory = os.path.join(BASE_DIRECTORY, 'data', posterior)

    fit = eos.data.ImportanceSamples(
        os.path.join(fit_directory, 'samples')
    )
    bound = eos.data.Prediction(
        os.path.join(
            fit_directory, 'pred-FF-fp-I1-dispersive-bound'
        )
    )
    sl = eos.data.ImportanceSamples(
        os.path.join(
            fit_directory, 'scattering-lengths', 'samples'
        )
    )
    radius_prediction = eos.data.Prediction(
        os.path.join(
            fit_directory,
            'pred-FF-fp-I1-pion-charge-radius-param',
        )
    )
    couplings = eos.data.ImportanceSamples(
        os.path.join(
            fit_directory, 'rho-couplings', 'samples'
        )
    )

    radius_values = charge_radius_squared_calculator(
        radius_prediction.samples[:, 0]
    )

    def summary(values, weights, digits=4, factor=1.0):
        values = np.asarray(values, dtype=float)

        median = factor * quantile_1D(values, weights, 0.50)
        lower = factor * quantile_1D(values, weights, 0.16)
        upper = factor * quantile_1D(values, weights, 0.84)

        return (
            f'{median:.{digits}f} '
            f'(+{upper - median:.{digits}f}'
            f'/-{median - lower:.{digits}f})'
        )

    # Evaluate goodness of fit at the saved mode, excluding root-penalty.
    full_analysis = af.analysis(posterior)
    mode = eos.data.Mode(
        os.path.join(fit_directory, 'mode-default')
    )

    best_fit_values = {
        parameter.name(): value
        for parameter, value in zip(
            full_analysis.varied_parameters, mode.mode
        )
    }

    for parameter in full_analysis.varied_parameters:
        full_analysis.parameters.set(
            parameter.name(),
            best_fit_values[parameter.name()],
        )
    full_gof = full_analysis.goodness_of_fit()

    reduced_args = dict(full_analysis.init_args)
    manual_constraints = dict(
        reduced_args.get('manual_constraints') or {}
    )

    root_names = [
        name for name in manual_constraints
        if str(name) == '0->pipi::root_penalty'
    ]
    if len(root_names) != 1:
        raise ValueError(
            f'Expected one root-penalty constraint; '
            f'found {root_names}.'
        )

    del manual_constraints[root_names[0]]
    reduced_args['manual_constraints'] = manual_constraints

    reduced_analysis = eos.Analysis(**reduced_args)
    for parameter in reduced_analysis.varied_parameters:
        reduced_analysis.parameters.set(
            parameter.name(),
            best_fit_values[parameter.name()],
        )

    chi2_without_root = (
        reduced_analysis.goodness_of_fit().total_chi_square()
    )
    dof_without_root = full_gof.total_degrees_of_freedom() - 1

    if dof_without_root <= 0:
        raise ValueError('Degrees of freedom must be positive.')

    pvalue = 100 * chi2.sf(
        chi2_without_root, dof_without_root
    )

    row = {
        'Fit': label,
        'p-value (%)': f'{pvalue:.2f}',

        'M_rho (MeV)': summary(
            fit.samples[:, 0], fit.weights,
            digits=2, factor=1000,
        ),
        'Gamma_rho (MeV)': summary(
            fit.samples[:, 1], fit.weights,
            digits=2, factor=1000,
        ),
        'Dispersive bound': summary(
            bound.samples[:, 0], bound.weights,
        ),
        'a11': summary(
            sl.samples[:, sl.lookup_table['derived::a11']],
            sl.weights,
        ),
        'b11': summary(
            sl.samples[:, sl.lookup_table['derived::b11']],
            sl.weights,
        ),
        'r_pi^2 (fm^2)': summary(
            radius_values, radius_prediction.weights,
        ),
        'Re(g_rhogamma)': summary(
            couplings.samples[
                :, couplings.lookup_table['derived::Re{g_rho_gamma}']
            ],
            couplings.weights, digits=3,
        ),
        'Im(g_rhogamma)': summary(
            couplings.samples[
                :, couplings.lookup_table['derived::Im{g_rho_gamma}']
            ],
            couplings.weights, digits=3,
        ),
        'Re(g_rhopipi)': summary(
            couplings.samples[
                :, couplings.lookup_table['derived::Re{g_rho_pi_pi}']
            ],
            couplings.weights, digits=3,
        ),
        'Im(g_rhopipi)': summary(
            couplings.samples[
                :, couplings.lookup_table['derived::Im{g_rho_pi_pi}']
            ],
            couplings.weights, digits=3,
        ),
    }

    return pd.DataFrame([row]).set_index('Fit')